# Producing `states/crystal_init.state` for Pokemon Crystal

Crystal needs a starting save-state — the equivalent of PWhiddy's `init.state` for Red. The save-state freezes the agent at a clean post-intro anchor so every training episode begins from the same place (no wasted steps on text boxes).

Two paths: scripted (recommended once it works) or manual (always works).

**Canonical anchor**: standing on the first walkable tile of **Cherrygrove City** facing south, name = `RED`, starter = `CYNDAQUIL`. This matches the convention we'll use in the smoke test.

## Option A — Scripted (C3b)

Once `scripts/make_crystal_init_state.py` ships (Phase IV C3b), one line:

```bash
python scripts/make_crystal_init_state.py \
    --rom roms/PokemonCrystal.gbc \
    --out states/crystal_init.state \
    --name RED \
    --starter cyndaquil
```

The script boots PyBoy headless, drives the intro with frame-precise inputs interleaved with RAM-state-driven waits (e.g., wait until `wMapGroup, wMapNumber` indicate Cherrygrove), saves the state, exits. ~2 minutes wall-time.

## Option B — Manual (always works)

If the script fails or you want a different anchor, do it by hand:

1. Install `pyboy` and `pillow` if you haven't: `pip install pyboy pillow`.
2. Run `python -c "from pyboy import PyBoy; p = PyBoy('roms/PokemonCrystal.gbc', window='SDL2'); p.set_emulation_speed(0); [p.tick(1, True) for _ in range(99999999)]"`. PyBoy opens the game in a visible window. `set_emulation_speed(0)` means "as fast as possible" — use `1` for real-time play. Keyboard controls: arrow keys + Z/A/Enter/Space ≈ A/B/Start/Select (PyBoy defaults).
3. Play through the intro: new game → name yourself `RED` (or whatever — recommend something short) → mom's dialog → Prof. Elm's lab → pick **Cyndaquil** (or your preferred starter — the adapter doesn't care) → walk out of New Bark Town → walk through Route 29 → arrive at Cherrygrove City. ~5 minutes.
4. Stand on the first walkable tile **just inside Cherrygrove** (facing south is conventional but optional).
5. In a separate Python REPL or by extending the script above:
   ```python
   from pathlib import Path
   Path('states').mkdir(exist_ok=True)
   with open('states/crystal_init.state', 'wb') as f:
       p.save_state(f)
   ```
6. Close PyBoy. The state file is ~150 KB.

Verify by loading it in the smoke test and inspecting `dump_state`:

In [ ]:
# Sanity-check the saved state
from deepEmulator.cartridges.pokemon_crystal import PokemonCrystalAdapter, dump_state
from deepEmulator.platforms.gameboy import PyBoyEnv

adapter = PokemonCrystalAdapter(init_state='states/crystal_init.state')
env = PyBoyEnv(adapter, rom_path='roms/PokemonCrystal.gbc',
               init_state='states/crystal_init.state', headless=True, max_steps=10)
env.reset()
print(dump_state(env.pyboy))
# Expected (approximate, depending on exact tile):
# party_count: 1 (just Cyndaquil)
# johto_badge_count: 0
# map_group: 26  (CHERRYGROVE_CITY group)
# map_number: 1  (or similar — within group 26)
# battle_mode: 0
# event_flags_set_total: small but nonzero (intro events fired)
env.close()

If `johto_badge_count > 0` or `party_count != 1` (assuming you picked exactly one starter), something is off. If `map_group, map_number` are way different from expected, the address constants in `pokemon_crystal.py` marked `# VERIFY` may need correction — compare the live values to the in-game observation and patch the constants.

---

## Pokemon Coral (Crystal-engine romhack)

Coral inherits Crystal's RAM layout, so `PokemonCoralAdapter` (a 10-line subclass) works directly. But Coral has a **different intro flow + different overworld maps**, so:

- The scripted `make_crystal_init_state.py` will **NOT** work for Coral — its RAM-state-driven waits target Crystal's Cherrygrove map IDs.
- The **manual path is the only option** for Coral right now: same procedure as Crystal Option B above, but using `roms/PokemonCoral.gbc` and saving to `states/coral_init.state`.

### Manual procedure for Coral

1. Boot Coral in a visible PyBoy window:
   ```bash
   python -c "from pyboy import PyBoy; p = PyBoy('roms/PokemonCoral.gbc', window='SDL2'); [p.tick(1, True) for _ in range(99999999)]"
   ```
   Keyboard: arrows + Z (A) + A (B) + Enter (Start) + Space (Select).
2. Play through Coral's intro: title → new game → whatever Coral's intro is → pick a starter → walk to the first overworld town/route past the starting area.
3. Stand on a **clean walkable tile** — avoid grass (random encounter risk) and ledges/warps.
4. In a Python REPL (separate terminal, or extend the script above):
   ```python
   from pathlib import Path
   Path('states').mkdir(exist_ok=True)
   with open('states/coral_init.state', 'wb') as f:
       p.save_state(f)
   ```
5. Upload `states/coral_init.state` to `/content/drive/MyDrive/deepEmulator/states/` so notebook 08 (Colab training) picks it up automatically.

**Time budget**: ~5 minutes of play. The difference between this 5-minute investment and skipping it is the difference between a Colab run that *learns* and a Colab run that's stuck in boot phase for 30 minutes with a flat reward curve.

In [ ]:
# Sanity-check a Coral save-state the same way:
from deepEmulator.cartridges import pokemon_coral  # registers POKEMON CORAL
from deepEmulator.cartridges.pokemon_crystal import dump_state
from deepEmulator.core import registry
from deepEmulator.platforms.gameboy import PyBoyEnv

adapter = registry.get('POKEMON CORAL')(init_state='states/coral_init.state')
env = PyBoyEnv(adapter, rom_path='roms/PokemonCoral.gbc',
               init_state='states/coral_init.state', headless=True, max_steps=10)
env.reset()
ram = dump_state(env.pyboy)
print(ram)
# Expected: party_count >= 1, johto_badge_count == 0, battle_mode == 0,
# event_flags_set_total > 0 (some intro events fired).
# Nonsense values (party_count=42, badges=255) → check the # VERIFY constants
# in pokemon_crystal.py: Coral may have shifted addresses.
env.close()